In [ ]:
import tenacity

retry = tenacity.retry(
    stop=tenacity.stop_after_attempt(12), 
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=64)
)

In [ ]:
import csv
from pathlib import Path

csv_file = "nikkeibp-may2000-abridged.csv"

urls_data = []

with open(csv_file, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    urls_data = list(reader)

In [ ]:
import time
import requests

@retry
def download_cdx_data(original_url): 
    base = "https://web.archive.org/cdx/search/cdx"
    params = {
        "url": original_url,
        "from": 20000501000000,
        "to": 20000531235959,
        "filter": "statuscode:200",
        "limit": "-1"
    }
    response = requests.get(base, params=params)
    time.sleep(2)
    response.raise_for_status()
    return response.text

In [ ]:
for url in urls_data: 
    cdx_file_path = Path(f"data/urls/{url['url']}/cdx.csv")
    if cdx_file_path.exists():
        print(f"CDX data for {url['url']} already exists at {cdx_file_path}. Skipping.")
        continue

    cdx_data = download_cdx_data(url['url'])
    cdx_file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cdx_file_path, 'w', encoding='utf-8') as cdx_file:
        cdx_file.write(cdx_data)
    print(f"CDX data saved for {url['url']} at {cdx_file_path}")

In [ ]:
@retry
def download_archived_snapshot(original_url, timestamp, rewrite_modifier="id_", sleep=1, use_apparent_encoding=True):
    snapshot_url = f"https://web.archive.org/web/{timestamp}{rewrite_modifier}/{original_url}"
    print(f"Fetching archived snapshot for: {snapshot_url}")
    response = requests.get(snapshot_url, allow_redirects=True, stream=True)
    print(f"Received response with status code: {response.status_code}")
    time.sleep(sleep)

    if response.status_code in [404, 403]: 
        return {
            "status_code": response.status_code,
            "payload": None,
            "headers": dict(response.headers)
        }
    response.raise_for_status()
    
    content_type = response.headers.get("Content-Type")

    if content_type and content_type.startswith("text"):
        if use_apparent_encoding:
            response.encoding = response.apparent_encoding
        return {
            "status_code": response.status_code,
            "payload": response.text,
            "headers": dict(response.headers)
        }
    else:
        return {
            "status_code": response.status_code,
            "payload": response.content,
            "headers": dict(response.headers)
        }
    

In [ ]:
from pathlib import Path
import csv

# find directories under data/urls not containing downloaded HTML files
url_dirs_to_download = [d for d in Path("data/urls").iterdir() if d.is_dir() and not any(f.suffix == ".html" for f in d.iterdir())]

for url_dir in url_dirs_to_download:
    cdx_file = url_dir / "cdx.csv"
    with open(cdx_file, 'r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=' ')
        rows = list(reader)
        if len(rows) == 0:
            print(f"No snapshot found in {cdx_file}. Skipping.")
            continue
        # since we requested one snapshot for each URL, we can just take the first row
        first_row = rows[0]
        if first_row:
            url, timestamp = first_row[2], first_row[1]
            html_download = download_archived_snapshot(original_url=url, timestamp=timestamp)
            html_content = html_download['payload']
            html_file_path = cdx_file.parent / f"{timestamp}.html"
            with open(html_file_path, 'w', encoding='utf-8') as html_file:
                html_file.write(html_content)
            print(f"{url} @ {timestamp} saved to {html_file_path}")

In [ ]:
# load banner ad dimensions from banner-ad-dimensions.csv
with open('banner-ad-dimensions.csv', 'r') as file:
    reader = csv.DictReader(file)
    banner_dimensions = list(reader)  # Convert to list to read all rows

from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup

def ensure_absolute_url(src, base_url):
    # Process the source URL to ensure it is absolute.
    # If the src is relative, it will be made absolute using the base URL.
    base_url = f'http://{base_url}' if not base_url.startswith(('http://')) else base_url
    if not (urlparse(src).netloc):
        # If src is relative, make it absolute using the base URL
        src = urljoin(base_url, src)
    elif src.startswith('//'):
        # If src is protocol-relative, add the HTTP scheme
        src = f"http:{src}"
    return src

def extract_ad_tags(html_content, html_base_url, banner_dimensions):
    soup = BeautifulSoup(html_content, 'html.parser')
    img_tags = soup.find_all('img')
    
    extracted_ads = []

    for img in img_tags:
        w = img.get('width')
        h = img.get('height')
        try:
            width = int(w)
            height = int(h)
        except (TypeError, ValueError):
            continue  # skip non-numeric or missing sizes

        for banner in banner_dimensions:
            if (width == int(banner['width']) and height == int(banner['height'])):
                src = img.get('src')
                src = ensure_absolute_url(src, html_base_url)
                ad_href = img.parent.get('href') if img.parent and img.parent.name == 'a' else None
                if ad_href:
                    ad_href = ensure_absolute_url(ad_href, html_base_url)
                alt = img.get('alt', '')
                extracted_ads.append({
                    'src': src,
                    'width': width,
                    'height': height,
                    'ad_href': ad_href,
                    'alt': alt
                })
                break  # stop checking once a match is found

    return extracted_ads



In [ ]:
# Walk through the data directory and process each HTML file
html_files = Path("data/urls/").glob("*/*.html")

all_extracted_ad_tags = []

for html_file in html_files:
    timestamp = html_file.stem
    base_url = html_file.parent.name 
    print(f"Processing HTML file: {html_file} with timestamp: {timestamp}")
    # read the HTML content
    with open(html_file, 'r', encoding='utf-8') as file:
        html_content = file.read()
    extracted_ad_tags = extract_ad_tags(html_content, base_url, banner_dimensions)
    for ad in extracted_ad_tags:
        ad['web_page_snapshot_timestamp'] = timestamp
        ad['web_page_original_url'] = base_url
    all_extracted_ad_tags.extend(extracted_ad_tags)

# Add a unique identifier to each ad tag
for idx, ad in enumerate(all_extracted_ad_tags):
    ad['id'] = idx + 1


In [ ]:
from datetime import datetime

def calculate_coherence(image_capture_date, image_last_modified_date, web_page_snapshot_timestamp):
    if not image_last_modified_date:
        return "Probably Violative"
    try:
        image_capture_dt = datetime.strptime(image_capture_date, '%a, %d %b %Y %H:%M:%S %Z')
        image_last_modified_dt = datetime.strptime(image_last_modified_date, '%a, %d %b %Y %H:%M:%S %Z')
        web_page_snapshot_dt = datetime.strptime(web_page_snapshot_timestamp, '%Y%m%d%H%M%S')
    except ValueError as e:
        raise ValueError(f"Date parsing error: {e}")

    if image_last_modified_dt <= web_page_snapshot_dt <= image_capture_dt:
        return "Prima Facie Coherent"
    elif web_page_snapshot_dt < image_last_modified_dt < image_capture_dt:
        return "Prima Facie Violative"
    elif image_last_modified_dt < image_capture_dt < web_page_snapshot_dt:
        return "Possibly Coherent"

In [ ]:
import json

for img_tag in all_extracted_ad_tags:
    id = img_tag.get('id')
    # check if the image has already been downloaded
    image_dir = Path(f"data/images/{id}/")
    image_data_path = image_dir / "image-data.json"
    
    if image_data_path.exists():
        print(f"Image data for ad ID {id} already exists at {image_data_path}.")
        continue

    src = img_tag['src']
    web_page_snapshot_timestamp = img_tag['web_page_snapshot_timestamp']

    download_image = download_archived_snapshot(src, web_page_snapshot_timestamp, rewrite_modifier="im_")

    headers = download_image["headers"]
    status_code = download_image["status_code"]
    content_type = headers.get("Content-Type", "")

    image_data_output = {
        "id": id,
        "image_src": src,
        "web_page_snapshot_timestamp": web_page_snapshot_timestamp,
        "web_page_original_url": img_tag.get('web_page_original_url'),
        "width": img_tag.get('width'),
        "height": img_tag.get('height'),
        "image_href": img_tag.get('ad_href'),
        "image_alt_text": img_tag.get('alt'),
        "image_snapshot_timestamp": None,
        "image_last_modified": None,
        "image_filetype": None,
        "status_code": status_code,
        "headers": headers,
        "coherence": None
    }
    
    if status_code == 200 and content_type.startswith("image"):
        image_filetype = content_type.split("/", 1)[-1] # this approach may not work for other file types
        image_file_path = image_dir / f"image.{image_filetype}"
        image_file_path.parent.mkdir(parents=True, exist_ok=True)

        image_data_output["image_filetype"] = image_filetype

        with open(image_file_path, "wb") as image_file:
            image_file.write(download_image["payload"])
        print(f"Image {id} downloaded and saved at {image_file_path}")

        # Retrieve headers for temporal coherence
        image_last_modified = headers.get("x-archive-orig-last-modified")
        image_capture_date = headers.get("memento-datetime")
        image_data_output["coherence"] = calculate_coherence(
            image_capture_date, image_last_modified, web_page_snapshot_timestamp
        )

        # Record parsed timestamps and filetype in yyyymmddhhmmss
        if image_capture_date:
            image_data_output["image_snapshot_timestamp"] = datetime.strptime(
                image_capture_date, "%a, %d %b %Y %H:%M:%S %Z"
            ).strftime("%Y%m%d%H%M%S")
        if image_last_modified:
            image_data_output["image_last_modified"] = datetime.strptime(
                image_last_modified, "%a, %d %b %Y %H:%M:%S %Z"
            ).strftime("%Y%m%d%H%M%S")
    else:
        print(f"Skipped ad ID {id}: status={status_code}, type={content_type!r}")

    # Save metadata regardless of download success
    image_data_path.parent.mkdir(parents=True, exist_ok=True)
    with open(image_data_path, "w", encoding="utf-8") as image_data_file:
        json.dump(image_data_output, image_data_file, ensure_ascii=False, indent=4)

In [ ]:
# consolidate the downloaded image data into a single JSON file
all_image_data = []
image_data_files = Path("data/images/").glob("*/image-data.json")
for image_data_file in image_data_files:
    with open(image_data_file, 'r', encoding='utf-8') as file:
        image_data = json.load(file)
        all_image_data.append(image_data)

# save consolidated image data to a single JSON file
with open('data/all_image_data.json', 'w', encoding='utf-8') as file:
    json.dump(all_image_data, file, ensure_ascii=False, indent=4)


In [ ]:
# OPTIONAL: print summary statistics
from collections import Counter

# Read the JSON file if not all_image_data:
if not all_image_data:
    with open('data/all_image_data.json', 'r', encoding='utf-8') as file:
        all_image_data = json.load(file)

# How many detected ad tags? 
print(f"Total detected ad tags: {len(all_image_data)}")

# How many images were not archived by (404) or excluded on (403) the Wayback Machine?
unarchived_images = [img for img in all_image_data if img['status_code'] == 404]
excluded_images = [img for img in all_image_data if img['status_code'] == 403]
print(f"Total images not archived (404): {len(unarchived_images)}")
print(f"Total images excluded (403): {len(excluded_images)}")

# How many ad image srcs were not downloaded due to non-image content type?
non_image_files = [img for img in all_image_data if img['status_code'] == 200 and not img['headers'].get('Content-Type', '').startswith('image')]
print(f"Total ad image srcs not downloaded due to non-image content type: {len(non_image_files)}")

# How many ad images were successfully downloaded?
successful_images = [img for img in all_image_data if img['status_code'] == 200 and img['headers'].get('Content-Type', '').startswith('image')]
print(f"Total ad images successfully downloaded: {len(successful_images)}")

# The distribution of coherence categories?
coherence_counter = Counter(img['coherence'] for img in all_image_data)
print("Coherence categories distribution:")
for category, count in coherence_counter.most_common():
    print(f"Category '{category}': {count} images")